# A GPU fluid simulator, explained in Python

This notebook is the runnable companion to the [project page](https://shauryaattreya37.github.io/High-Performance-LBM-On-GPU-/). You run the project's lattice Boltzmann solver yourself, check it against exact answers and published data, and explore the GPU results measured for the page. The solver code was written by Claude, Anthropic's AI, under Shaurya Attreya's direction.

**No GPU is needed.** Everything here runs on the NumPy reference solver, the same physics as the GPU kernels. Grids are kept small so that it stays quick.

Two kinds of result appear, and each section says which it uses:

- **Run here:** computed on your machine when you run the cell.
- **Stored:** measured earlier on an RTX 3050 laptop GPU, and only plotted here.

Choose **Runtime > Run all**. It should take a few minutes. The charts are interactive: hover for values, click a legend entry to hide a curve, and move the sliders.

In [ ]:
%pip install -q "git+https://github.com/ShauryaAttreya37/High-Performance-LBM-On-GPU-.git" plotly

In [ ]:
import time

import matplotlib
import numpy as np
import plotly.graph_objects as go
from IPython.display import Image as show_image
from PIL import Image

from lbm import SimulationParams, make_solver
from lbm.boundaries import cavity_walls
from lbm.cases.ghia1982 import ghia_centerlines
from lbm.cases.lid_driven_cavity import run_cavity
from lbm.cases.poiseuille import run_poiseuille
from lbm.cases.taylor_green import convergence_order, run_taylor_green
from lbm.diagnostics import vorticity
from lbm.lattice import D2Q9

BLUE, ORANGE, GREEN, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#898781"


def chart(title, x_title, y_title, log_x=False, log_y=False, height=420):
    "An empty interactive chart; every plot below starts from this one."
    figure = go.Figure()
    figure.update_layout(
        title=title, template="plotly_white", height=height, hovermode="closest",
        xaxis_title=x_title, yaxis_title=y_title,
        xaxis_exponentformat="e", yaxis_exponentformat="e",
    )
    # Left unset otherwise, so that plotly can use named categories on the axis.
    if log_x:
        figure.update_xaxes(type="log")
    if log_y:
        figure.update_yaxes(type="log")
    return figure


def curve(figure, x, y, name, color, symbol="circle"):
    "Add one measured curve, drawn with a line and markers."
    figure.add_trace(go.Scatter(
        x=x, y=y, name=name, mode="lines+markers",
        line={"color": color, "width": 2}, marker={"color": color, "size": 9, "symbol": symbol},
    ))


def order_two_guide(figure, x, y_last):
    "Add a dashed line showing an ideal second-order trend, ending at (x[-1], y_last)."
    x = np.asarray(x, dtype=float)
    figure.add_trace(go.Scatter(
        x=x, y=y_last * (x[-1] / x) ** 2, name="Second-order guide", mode="lines",
        line={"color": GREY, "width": 1.5, "dash": "dash"}, hoverinfo="skip",
    ))

## 1. The physics in four ideas

**Populations.** Each grid point stores numbers $f_i$: the amount of fluid moving in one allowed direction $\mathbf{c}_i$. In 2D the solver uses D2Q9, nine directions with fixed weights $w_i$. Density and velocity are just sums over the populations:

$$\rho = \sum_i f_i, \qquad \rho\,\mathbf{u} = \sum_i \mathbf{c}_i f_i$$

**Streaming.** In each step, every population moves one grid point along its direction.

**Collision.** At each point, populations then relax towards a local equilibrium with the same density and momentum, at a rate set by the relaxation time $\tau$.

$$f_i^{*} = f_i - \frac{1}{\tau}\left(f_i - f_i^{\mathrm{eq}}\right)$$

**Back to fluid.** At low speed these two steps reproduce the Navier–Stokes equations, with viscosity $\nu = \tfrac{1}{3}\left(\tau - \tfrac{1}{2}\right)$.

Here is the lattice the solver really uses. For the method to conserve momentum and behave like a fluid, the weights must sum to 1, and the mean velocity must be zero.

In [ ]:
for direction, (velocity, weight) in enumerate(zip(D2Q9.c, D2Q9.w)):
    print(f"c_{direction} = {str(velocity):9}  weight = {weight:.4f}")

weights = np.array(D2Q9.w)
velocities = np.array(D2Q9.c)
print(f"\nSum of weights: {weights.sum():.12f}")
print(f"Mean velocity:  {(weights[:, None] * velocities).sum(axis=0)}")

## 2. Channel flow against the exact answer

*Run here.* A body force pushes fluid between two walls, and theory says the velocity profile is a parabola. The solver has two collision models: simple BGK and TRT, which relaxes two parts of the populations separately.

TRT should put the wall exactly where it belongs for any $\tau$. In BGK, the wall's effective position shifts with $\tau$. This cell runs both across a range of $\tau$ to test that claim.

In [ ]:
TAUS = [0.6, 0.7, 0.8, 1.0, 1.25, 1.5, 2.0]
channel_runs = {
    collision: [run_poiseuille(height=16, tau=tau, collision=collision) for tau in TAUS]
    for collision in ("bgk", "trt")
}

figure = chart("Channel flow: error against the exact parabola", "relaxation time τ",
               "relative L₂ error", log_y=True)
curve(figure, TAUS, [run.l2_error for run in channel_runs["bgk"]], "BGK", ORANGE, "diamond")
curve(figure, TAUS, [run.l2_error for run in channel_runs["trt"]], "TRT", BLUE)
figure.show()

TRT sits near round-off for every $\tau$. BGK's error comes from its wall position, so it changes with $\tau$.

Use the slider to see the error across the channel at each $\tau$.

In [ ]:
figure = chart("Channel flow: error across the channel", "distance from lower wall (nodes)",
               "|u − u_exact| / u_max", log_y=True)
for index, tau in enumerate(TAUS):
    for collision, color in (("bgk", ORANGE), ("trt", BLUE)):
        run = channel_runs[collision][index]
        # The small offset keeps exact zeros visible on a log axis.
        error = np.abs(run.u - run.u_exact) / run.u_exact.max() + 1e-17
        figure.add_trace(go.Scatter(
            x=run.y, y=error, name=collision.upper(), mode="lines+markers",
            line={"color": color, "width": 2}, marker={"color": color, "size": 7},
            visible=index == 2,
        ))

steps = [
    {"label": str(tau), "method": "update",
     "args": [{"visible": [i // 2 == index for i in range(2 * len(TAUS))]}]}
    for index, tau in enumerate(TAUS)
]
figure.update_layout(
    sliders=[{"active": 2, "currentvalue": {"prefix": "τ = "}, "pad": {"t": 60}, "steps": steps}],
    height=540, yaxis_range=[-17.5, -1],
)
figure.show()

## 3. Does the error fall as theory predicts?

*Run here.* The Taylor–Green vortex is a decaying flow with an exact solution. The method is second order, so doubling the grid should cut the error by four. On these log axes, that is a straight line parallel to the dashed guide.

In [ ]:
vortex_runs = [run_taylor_green(n=n, tau=0.8) for n in (16, 32, 64, 128)]
orders = convergence_order(vortex_runs)
print("Observed order per doubling:", ", ".join(f"{order:.2f}" for order in orders))

grid_sizes = [run.n for run in vortex_runs]
errors = [run.l2_error for run in vortex_runs]
figure = chart("Taylor–Green vortex: error against grid size", "grid size N", "relative L₂ velocity error",
               log_x=True, log_y=True)
curve(figure, grid_sizes, errors, "This solver, run here", BLUE)
order_two_guide(figure, grid_sizes, errors[-1])
figure.show()

## 4. Lid-driven cavity against published data

*Run here.* A lid slides across a box of fluid and drives a large vortex. This compares the result with the classic data of Ghia et al. (1982) at Re = 100.

In [ ]:
cavity = run_cavity(reynolds=100, n=64)
reference = ghia_centerlines(100)
print(f"Steps to steady state: {cavity.steps:,}")
print(f"Largest centreline deviation from Ghia, in units of lid speed: {cavity.max_error:.4f}")

figure = chart("Cavity centrelines against Ghia et al. (1982), Re = 100", "position along centreline",
               "velocity / lid speed")
figure.add_trace(go.Scatter(x=cavity.y, y=cavity.u, name="u on vertical centreline", mode="lines",
                            line={"color": BLUE, "width": 2}))
figure.add_trace(go.Scatter(x=cavity.x, y=cavity.v, name="v on horizontal centreline", mode="lines",
                            line={"color": ORANGE, "width": 2}))
figure.add_trace(go.Scatter(x=reference.y, y=reference.u, name="Ghia u", mode="markers",
                            marker={"color": BLUE, "size": 8, "symbol": "circle-open", "line": {"width": 2}}))
figure.add_trace(go.Scatter(x=reference.x, y=reference.v, name="Ghia v", mode="markers",
                            marker={"color": ORANGE, "size": 8, "symbol": "diamond-open", "line": {"width": 2}}))
figure.show()

### Watch it spin up

*Run here.* This cell runs a cavity from rest at Re = 400, saves a frame after each lid-transit time, and writes them to a GIF. Red and blue show fluid spinning in opposite directions. The solver calls are the same ones `run_cavity` uses.

In [ ]:
N, LID_SPEED, REYNOLDS, FRAMES = 64, 0.1, 400, 40
solid, wall = cavity_walls(N, LID_SPEED)
params = SimulationParams(
    nx=N + 2, ny=N + 2, tau=3.0 * LID_SPEED * N / REYNOLDS + 0.5,
    collision="trt", solid=solid, wall_velocity=wall,
)
solver = make_solver(params, backend="numpy")
solver.initialize()
colormap = matplotlib.colormaps["RdBu_r"]

frames = []
for _ in range(FRAMES):
    solver.run(2 * round(N / LID_SPEED / 2))
    fields = solver.macroscopic()
    spin = vorticity(fields.ux[1:-1, 1:-1], fields.uy[1:-1, 1:-1]) * N / LID_SPEED
    colours = colormap(np.clip(spin / 10 + 0.5, 0, 1), bytes=True)[::-1, :, :3]
    frames.append(Image.fromarray(colours).resize((384, 384), Image.Resampling.NEAREST))

frames[0].save("cavity_spinup.gif", save_all=True, append_images=frames[1:], duration=120, loop=0)
show_image("cavity_spinup.gif")

## 5. Why high Reynolds numbers are hard

*A formula, no simulation.* Combining $\mathrm{Re} = uN/\nu$ with the viscosity formula gives $\tau - \tfrac12 = 3uN/\mathrm{Re}$, for a body $N$ nodes long and lattice speed $u$. As $\tau$ approaches $\tfrac12$ the simple scheme becomes unstable, and the solver warns below $\tau = 0.51$.

Move the slider to see how many nodes it takes to stay above that level. This is one reason the project's goal of drag predictions up to Re = 100,000 remains unfinished.

In [ ]:
LATTICE_SPEED = 0.05
GRID_SIZES = [64, 128, 256, 512, 1024, 2048]
reynolds_numbers = np.logspace(1, 5, 41)

figure = chart(f"How close τ gets to ½, at u = {LATTICE_SPEED}", "Reynolds number", "τ − ½",
               log_x=True, log_y=True, height=540)
for nodes in GRID_SIZES:
    figure.add_trace(go.Scatter(
        x=reynolds_numbers, y=3 * LATTICE_SPEED * nodes / reynolds_numbers, name=f"N = {nodes}",
        mode="lines", line={"color": BLUE, "width": 2}, visible=nodes == 256,
    ))
figure.add_hline(y=0.01, line_color=GREY)
# On a log axis, plotly places annotations by the logarithm of the value: 0.01 is -2.
figure.add_annotation(text="τ = 0.51: solver warns", xref="x domain", x=1, y=-2,
                      xanchor="right", yshift=10, showarrow=False)
steps = [
    {"label": str(nodes), "method": "update",
     "args": [{"visible": [other == nodes for other in GRID_SIZES]}]}
    for nodes in GRID_SIZES
]
figure.update_layout(
    sliders=[{"active": 2, "currentvalue": {"prefix": "Nodes along the body: "}, "pad": {"t": 60},
              "steps": steps}],
    yaxis_range=[-4, 2], showlegend=False,
)
figure.show()

## 6. Making it fast: your CPU against the GPU

*Run here, then stored.* LBM moves a lot of data and does little arithmetic per point, so memory bandwidth sets the speed limit. A 2D float32 update reads and writes nine values, or 72 bytes. The RTX 3050's measured 156 GB/s therefore caps it near 2,170 million lattice updates per second (MLUPS).

This cell times the NumPy solver on your machine in float32, on a smaller grid than the GPU benchmark.

In [ ]:
SIZE, STEPS = 512, 40
params = SimulationParams(nx=SIZE, ny=SIZE, tau=0.6, collision="trt", dtype="float32")
solver = make_solver(params, backend="numpy")
solver.initialize()

start = time.perf_counter()
solver.run(STEPS)
your_mlups = SIZE**2 * STEPS / (time.perf_counter() - start) / 1e6
print(f"NumPy on this machine: {your_mlups:.1f} million updates per second")
print(f"The fused GPU kernel at 2048² is about {2141 / your_mlups:,.0f} times faster.")

The bars below are stored measurements, apart from your own. The kernels are the three GPU designs from the project: a **tiled** kernel using on-chip shared memory, the in-place **AA** pattern using half the memory, and a **two-lattice** kernel.

In [ ]:
# Stored: benchmarks/bench_mlups.py on an RTX 3050 laptop GPU, float32, 2048² unless noted.
# NumPy (1024²) and CuPy come from earlier runs than the three kernels.
speeds = {
    "Your NumPy run (512²)": your_mlups,
    "NumPy, reference CPU (1024²)": 6,
    "CuPy arrays": 57,
    "Tiled kernel": 1996,
    "AA in-place kernel": 2123,
    "Two-lattice kernel": 2141,
}
figure = chart("2D speed, float32", "", "million updates per second", log_y=True)
figure.add_trace(go.Bar(x=list(speeds), y=list(speeds.values()), marker_color=BLUE,
                        text=[f"{value:,.0f}" for value in speeds.values()], textposition="outside"))
figure.update_yaxes(range=[0, 4])
figure.show()

In [ ]:
# Stored: share of measured memory bandwidth used, 2D float32.
grid_labels = ["1024²", "2048²", "4096²"]
figure = chart("How close each kernel gets to the memory limit", "grid size", "% of peak bandwidth")
curve(figure, grid_labels, [98.7, 98.7, 96.3], "Two-lattice", BLUE)
curve(figure, grid_labels, [98.3, 97.8, 94.9], "AA in-place", ORANGE, "diamond")
curve(figure, grid_labels, [95.0, 92.0, 82.3], "Tiled", GREEN, "triangle-up")
figure.update_yaxes(range=[80, 100])
figure.show()

# Stored: 3D, D3Q19, float32. One update moves 152 bytes, so the limit is about 1,026.
grid_labels = ["64³", "96³", "128³", "192³", "256³"]
figure = chart("3D speed", "grid size", "million updates per second")
curve(figure, grid_labels, [1021, 1022, 1015, 1005, 991], "Two-lattice", BLUE)
curve(figure, grid_labels, [1014, 1012, 1000, 986, 957], "AA in-place", ORANGE, "diamond")
figure.add_hline(y=156e9 / 152 / 1e6, line_color=GREY, annotation_text="memory limit")
figure.update_yaxes(range=[900, 1050])
figure.show()

## 7. Curved bodies and a hidden error

*Stored.* Plain bounce-back turns a circle into grid-shaped stairs. Bouzidi's method uses how far along each grid link the true surface sits, and the immersed boundary method (IBM) has surface points push on the fluid. These problems have known answers, so lower is better.

In [ ]:
# Stored: lbm.cases.stokes_array and lbm.cases.pipe_flow, RTX 3050, 2026-10-27.
radii = [4.04, 6.06, 8.07, 12.1, 16.15, 24.2]
figure = chart("2D cylinder array: drag error vs Sangani–Acrivos", "cylinder radius (nodes)",
               "|drag error| (%)", log_x=True, log_y=True)
curve(figure, radii, [0.562, 2.654, 1.154, 1.244, 0.449, 0.315], "Staircase", ORANGE, "diamond")
curve(figure, radii[:4], [12.588, 7.533, 5.357, 3.384], "IBM", GREEN, "triangle-up")
curve(figure, radii, [0.723, 0.178, 0.101, 0.034, 0.038, 0.001], "Bouzidi", BLUE)
figure.show()

radii = [4.3, 6.3, 8.3, 12.3, 16.3, 24.3]
bouzidi = [9.658e-3, 1.025e-2, 2.497e-3, 7.107e-4, 5.875e-4, 1.873e-4]
figure = chart("3D pipe flow: error vs the exact profile", "pipe radius (nodes)", "relative L₂ error",
               log_x=True, log_y=True)
curve(figure, radii, [6.304e-2, 9.867e-2, 3.362e-2, 1.576e-2, 1.383e-2, 8.626e-3], "Staircase", ORANGE, "diamond")
curve(figure, radii, bouzidi, "Bouzidi", BLUE)
order_two_guide(figure, radii, bouzidi[-1])
figure.show()

**A drag error that better resolution didn't fix.** Cylinder drag came out about 2 % too high, and a finer grid didn't help, which pointed away from grid error. LBM is slightly compressible, with an error that grows with the square of the Mach number. So plotting drag against the square of the lattice speed should give a straight line, and this cell fits one to the stored results. Where the line meets zero speed is the drag with that error removed.

In [ ]:
# Stored: lbm.cases.cylinder_benchmark --test 2D-1 at N = 40, three lattice speeds.
speeds_squared = np.array([0.075, 0.0375, 0.01875]) ** 2
drag = np.array([5.694, 5.609, 5.589])
slope, intercept = np.polyfit(speeds_squared, drag, 1)
print(f"Drag extrapolated to zero Mach number: {intercept:.2f} (accepted interval 5.57–5.59)")

fit_x = np.array([0, 0.006])
figure = chart("Cylinder drag against squared lattice speed", "u_max² (lattice units)", "drag coefficient C_D")
figure.add_hrect(y0=5.57, y1=5.59, fillcolor=GREEN, opacity=0.18, line_width=0,
                 annotation_text="accepted interval")
figure.add_trace(go.Scatter(x=fit_x, y=intercept + slope * fit_x, name="Straight-line fit", mode="lines",
                            line={"color": GREY, "width": 1.5, "dash": "dash"}, hoverinfo="skip"))
figure.add_trace(go.Scatter(x=speeds_squared, y=drag, name="Measured", mode="markers",
                            marker={"color": BLUE, "size": 11}))
figure.show()

## 8. Your turn

*Run here.* Change the values and rerun the cell. Viscosity is $\nu = (\tau - 1/2)/3$. Values of `tau` close to 0.5 make the channel slower to settle, and `reynolds` can be 100, 400 or 1000. A cavity at 1000 takes much longer, so start with `n = 64` at 100 or 400.

At Re = 400 the cell prints two numbers, because one reference value stands out. Ghia's listed $v = -0.23827$ at $x = 0.9063$ breaks the smooth trend of its neighbours, and the gap there does not shrink on finer grids, unlike every other point. So it can dominate the largest deviation, and the next-largest is the better guide.

In [ ]:
tau = 1.2
for collision in ("trt", "bgk"):
    result = run_poiseuille(height=16, tau=tau, collision=collision)
    print(f"Channel, {collision}, tau = {tau}: relative L2 error {result.l2_error:.2e}")

reynolds, n = 400, 64
cavity = run_cavity(reynolds=reynolds, n=n)
deviations = np.sort(np.abs(np.concatenate(cavity.errors_vs_ghia())))
print(f"Cavity, Re = {reynolds}, n = {n}: deviation from Ghia, in units of lid speed:")
print(f"  largest {deviations[-1]:.4f}, next largest {deviations[-2]:.4f}")

figure = go.Figure(go.Heatmap(z=cavity.vorticity, colorscale="RdBu_r", zmid=0, zmin=-10, zmax=10,
                              colorbar={"title": "vorticity"}))
figure.update_layout(title=f"Cavity vorticity, Re = {reynolds}", height=520, width=580,
                     yaxis={"scaleanchor": "x", "constrain": "domain"}, xaxis={"constrain": "domain"})
figure.show()

## Where next

- The [project page](https://shauryaattreya37.github.io/High-Performance-LBM-On-GPU-/) has the full explanation, the 3D results and the project's limits.
- The [repository](https://github.com/ShauryaAttreya37/High-Performance-LBM-On-GPU-) has the solver, GPU kernels and tests. The GPU kernels need an NVIDIA GPU, and the README shows how to run them.